In [ ]:
Leaf disease detection and recommendation project.

In [ ]:
Step 1: isntalling all the libraries and doing the text pre-processing and image dataset exploration.

In [12]:
pip install torch torchvision numpy scikit-learn matplotlib seaborn streamlit huggingface_hub pillow

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
import pandas as pd
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

def perform_dataset_eda(base_data_dir):
    """Explores directory topology and logs Kaggle class distribution profiles."""
    train_dir = os.path.join(base_data_dir, "train")
    valid_dir = os.path.join(base_data_dir, "valid")
    test_dir = os.path.join(base_data_dir, "test")
    
    print("\n📊 --- KAGGLE NEW PLANT DISEASES DATASET EDA ---")
    
    splits = [("Training (70%)", train_dir), ("Validation (15%)", valid_dir), ("Testing (15%)", test_dir)]
    for split_name, split_path in splits:
        if not os.path.exists(split_path):
            print(f"❌ Structural Path Missing: {split_path}")
            continue
            
        classes = sorted(os.listdir(split_path))
        total_images = 0
        class_stats = []
        
        for c in classes:
            c_path = os.path.join(split_path, c)
            if os.path.isdir(c_path):
                img_count = len([f for f in os.listdir(c_path) if f.lower().endswith(('png', 'jpg', 'jpeg'))])
                total_images += img_count
                class_stats.append({"Class": c, "Count": img_count})
                
        print(f"📈 {split_name} Split: Found {len(classes)} classes containing {total_images} total leaf images.")
        if "Training" in split_name and class_stats:
            df = pd.DataFrame(class_stats)
            print(f"   ↳ Class Size Range: Min={df['Count'].min()} | Max={df['Count'].max()} images per folder.")

def get_desktop_data_loaders(batch_size=32):
    """
    Steps 1, 2 & 3: References existing train, valid, and test folders 
    from Desktop, applying standard transformations and augmentations.
    """
    desktop_path = os.path.expanduser("~/Desktop")
    
    base_data_dir = os.path.join(desktop_path, "finalproject")
    train_dir = os.path.join(base_data_dir, "train")
    valid_dir = os.path.join(base_data_dir, "valid")
    test_dir = os.path.join(base_data_dir, "test")
    
    perform_dataset_eda(base_data_dir)

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),                                 
        transforms.RandomRotation(degrees=25),                        
        transforms.RandomResizedCrop((224, 224), scale=(0.75, 1.0)),  
        transforms.RandomHorizontalFlip(p=0.5),                       
        transforms.RandomVerticalFlip(p=0.5),                        
        transforms.ColorJitter(brightness=0.25, contrast=0.2),       
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    if not os.path.exists(train_dir) or not os.path.exists(valid_dir) or not os.path.exists(test_dir):
        fallback_classes = ["Tomato___Early_blight", "Potato___Early_blight", "Corn___Common_rust", "Healthy_Leaf"]
        return None, None, None, len(fallback_classes), fallback_classes

    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
    
    master_class_to_idx = train_dataset.class_to_idx
    class_names = train_dataset.classes
    num_classes = len(class_names)
    
    val_dataset = datasets.ImageFolder(root=valid_dir, transform=val_test_transform)
    test_dataset = datasets.ImageFolder(root=test_dir, transform=val_test_transform)
    
    val_dataset.class_to_idx = master_class_to_idx
    val_dataset.samples = [
        (path, master_class_to_idx[os.path.basename(os.path.dirname(path))]) 
        for path, _ in val_dataset.samples 
        if os.path.basename(os.path.dirname(path)) in master_class_to_idx
    ]
    
    test_dataset.class_to_idx = master_class_to_idx
    test_dataset.samples = [
        (path, master_class_to_idx[os.path.basename(os.path.dirname(path))]) 
        for path, _ in test_dataset.samples 
        if os.path.basename(os.path.dirname(path)) in master_class_to_idx
    ]
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, test_loader, num_classes, class_names

In [ ]:
Step 2: constructing a baseline cnn architecture model and transfer learning models.

In [15]:
import torch
import torch.nn as nn
import torchvision.models as models

class BaselineCNN(nn.Module):
    """Custom Baseline CNN Model Architecture."""
    def __init__(self, num_classes):
        super(BaselineCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),      
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                             
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),    
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                            
            nn.Dropout(0.25)                               
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))
        self.flatten = nn.Flatten()
        
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),                     
            nn.ReLU(),
            nn.Dropout(0.5),                               
            nn.Linear(128, num_classes)                    
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        return self.classifier(x)

def load_transfer_model(model_name="MobileNetV2", num_classes=10):
    """Compiles professional deep networks swapping standard classifiers."""
    if model_name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        
    # =======================================================================
    # 🎯 UPDATED EFFICIENTNET INDEXING LAYER HERE
    # =======================================================================
    elif model_name == "EfficientNetB0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        # Safely query out_features from the internal linear block layer
        in_features = m.classifier.in_features 
        m.classifier = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(in_features, num_classes)
        )
    # =======================================================================
    
    else: 
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        m.classifier = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(m.last_channel, num_classes)
        )
    return m

In [ ]:
Step 3: implementing genai using the hugging face. 

In [5]:
pip install --upgrade ipywidgets jupyterlab_widgets widgetsnbextension

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------- ----------------- 524.3/914.9 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------- 914.9/914.9 kB 2.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 3.3 MB/s eta 0:00:01
   ----------------------- ---------------- 1.3/2.2 MB 3.2 MB/s eta 0:00:01
   --------------------------------- ------ 1.8/2.2 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 3.2 MB/s  0:00:00

   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from huggingface_hub import InferenceClient

def query_treatment_advisory(predicted_disease, hf_token=None):
    """Step 6: Formulates customized system instructions to fetch structured advice."""
    if "healthy" in predicted_disease.lower():
        return "### Leaf Status: Healthy\nNo pathogenic targets found. Maintain standard irrigation parameters."

    model_endpoint = "Qwen/Qwen2.5-7B-Instruct"
    token_str = hf_token if (hf_token and hf_token != "YOUR_HF_TOKEN") else None
    
    try:
        client = InferenceClient(model=model_endpoint, token=token_str)
        clean_name = predicted_disease.replace("___", " ").replace("_", " ")
        
        messages = [
            {"role": "system", "content": "You are a senior plant pathologist. Output response structured inside clean markdown subsections without pleasantries."},
            {"role": "user", "content": f"Provide an organic or chemical treatment report for crop fields diagnosed with leaf disease: {clean_name}."}
        ]
        
        response = client.chat_completion(messages=messages, max_tokens=500, temperature=0.2)
        return response.choices[0].message.content
        
    except Exception as e:
        return (
            "### 📋 Advisory Error Fallback Summary\n"
            "**Treatment Recommendation**: Prune and isolate visual lesions. Apply standard broad-spectrum copper fungicide suspension."
        )

In [ ]:
Step 4: using all the above steps to build the entire model with deep learning cnn architecture and implementing the evaluation metrics on the test dataset. 

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from huggingface_hub import InferenceClient

def perform_dataset_eda(base_data_dir):
    """Explores directory topology and logs Kaggle class distribution profiles."""
    train_dir = os.path.join(base_data_dir, "train")
    valid_dir = os.path.join(base_data_dir, "valid")
    test_dir = os.path.join(base_data_dir, "test")
    
    print("\n📊 --- KAGGLE NEW PLANT DISEASES DATASET EDA ---")
    
    splits = [("Training (70%)", train_dir), ("Validation (15%)", valid_dir), ("Testing (15%)", test_dir)]
    for split_name, split_path in splits:
        if not os.path.exists(split_path):
            print(f"❌ Structural Path Missing: {split_path}")
            continue
            
        classes = sorted(os.listdir(split_path))
        total_images = 0
        class_stats = []
        
        for c in classes:
            c_path = os.path.join(split_path, c)
            if os.path.isdir(c_path):
                img_count = len([f for f in os.listdir(c_path) if f.lower().endswith(('png', 'jpg', 'jpeg'))])
                total_images += img_count
                class_stats.append({"Class": c, "Count": img_count})
                
        print(f"📈 {split_name} Split: Found {len(classes)} classes containing {total_images} total leaf images.")
        if "Training" in split_name and class_stats:
            df = pd.DataFrame(class_stats)
            print(f"   ↳ Class Size Range: Min={df['Count'].min()} | Max={df['Count'].max()} images per folder.")

def get_desktop_data_loaders(batch_size=32):
    """
    References existing train, valid, and test folders from Desktop,
    applying standard transformations and enforcing synchronized class maps.
    """
    desktop_path = os.path.expanduser("~/Desktop")
    base_data_dir = os.path.join(desktop_path, "finalproject")
    train_dir = os.path.join(base_data_dir, "train")
    valid_dir = os.path.join(base_data_dir, "valid")
    test_dir = os.path.join(base_data_dir, "test")
    
    perform_dataset_eda(base_data_dir)

    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomRotation(degrees=25),
        transforms.RandomResizedCrop((224, 224), scale=(0.75, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.25, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    if not os.path.exists(train_dir) or not os.path.exists(valid_dir):
        fallback_classes = ["Tomato___Early_blight", "Potato___Early_blight", "Corn___Common_rust", "Healthy_Leaf"]
        return None, None, None, len(fallback_classes), fallback_classes

    train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
    master_class_to_idx = train_dataset.class_to_idx
    class_names = train_dataset.classes
    num_classes = len(class_names)
    
    val_dataset = datasets.ImageFolder(root=valid_dir, transform=val_test_transform)
    val_dataset.class_to_idx = master_class_to_idx
    val_dataset.samples = [(p, master_class_to_idx[os.path.basename(os.path.dirname(p))]) 
                           for p, _ in val_dataset.samples if os.path.basename(os.path.dirname(p)) in master_class_to_idx]
    

    try:
        if not os.path.exists(test_dir):
            raise FileNotFoundError()
            
        test_dataset = datasets.ImageFolder(root=test_dir, transform=val_test_transform)
        test_dataset.class_to_idx = master_class_to_idx
        test_dataset.samples = [(p, master_class_to_idx[os.path.basename(os.path.dirname(p))]) 
                                for p, _ in test_dataset.samples if os.path.basename(os.path.dirname(p)) in master_class_to_idx]
    except (FileNotFoundError, RuntimeError):
        print("\n⚠️  [PIPELINE NOTICE]: Empty/missing 'test' class taxonomy detected.")
        print("   ↳ Duplicating 'valid' partition arrays to bypass testing target generation crash.")
        test_dataset = val_dataset
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    return train_loader, val_loader, test_loader, num_classes, class_names

class BaselineCNN(nn.Module):
    """Custom Baseline CNN Model Architecture."""
    def __init__(self, num_classes):
        super(BaselineCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))
        self.flatten = nn.Flatten()
        
        self.classifier = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.adaptive_pool(x)
        x = self.flatten(x)
        return self.classifier(x)

def load_transfer_model(model_name="MobileNetV2", num_classes=10):
    """Compiles professional deep networks swapping standard classifiers."""
    if model_name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == "EfficientNetB0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = m.classifier.in_features
        m.classifier = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(in_features, num_classes)
        )
    else:
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        m.classifier = nn.Sequential(
            nn.Dropout(p=0.2, inplace=True),
            nn.Linear(m.last_channel, num_classes)
        )
    return m


def comprehensive_eval(model, loader, device, model_name="Target Model", classes=None):
    """Calculates evaluation metrics and traces misclassified items."""
    if loader is None or len(loader.dataset) == 0:
        print(f"\n❌ EVALUATION ERROR: The data loader for '{model_name}' has 0 samples.")
        print("💡 Action required: Check that folder names in your 'test' or 'valid' directories")
        print("   match the exact spelling and casing (capitalisation) of your 'train' folder names.")
        return pd.DataFrame(columns=["Category", "Metric Name", "Value"])


    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
            
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    
    if classes is None:
        classes = [f"Class {i}" for i in range(cm.shape[0])]
        
    print(f"\n==================== 📊 STEP 5 METRICS EVALUATION: {model_name} ====================")
    print(f"✔️ Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f} | F1-Score: {f1:.4f}\n")
    
    print("🎯 --- PER-CLASS ACCURACY TRACKING ---")
    cm_acc = cm.diagonal() / (cm.sum(axis=1) + 1e-9)
    cm_acc_dict = {}
    for idx in range(len(cm_acc)):
        c_name = classes[idx] if idx < len(classes) else f"Index Missing Class Label #{idx}"
        print(f" 🔸 {c_name}: {cm_acc[idx]*100:.2f}%")
        cm_acc_dict[c_name] = cm_acc[idx]
        
    misclassified_indices = np.where(all_preds != all_labels)[0]
    print(f"\n❌ Total Misclassified Samples on split: {len(misclassified_indices)} items.")
    print("===========================================================================\n")
    
    report_rows = [
        {"Category": "Overall Model", "Metric Name": "Accuracy", "Value": acc},
        {"Category": "Overall Model", "Metric Name": "Precision (Weighted)", "Value": prec},
        {"Category": "Overall Model", "Metric Name": "Recall (Weighted)", "Value": rec},
        {"Category": "Overall Model", "Metric Name": "F1-Score (Weighted)", "Value": f1},
        {"Category": "Overall Model", "Metric Name": "Total Misclassifications", "Value": float(len(misclassified_indices))}
    ]
    for c_name, c_acc in cm_acc_dict.items():
        report_rows.append({"Category": f"Class Breakdown: {c_name}", "Metric Name": "Accuracy Score", "Value": c_acc})
        
    df_metrics = pd.DataFrame(report_rows)
    df_metrics["Value"] = df_metrics["Value"].round(4)
    return df_metrics


def query_treatment_advisory(predicted_disease, hf_token=None):
    """Formulates customized instructions to fetch structured advice from Qwen."""
    if "healthy" in predicted_disease.lower():
        return "### Leaf Status: Healthy\nNo pathogenic targets found. Maintain standard irrigation parameters."

    model_endpoint = "Qwen/Qwen2.5-7B-Instruct"
    token_str = hf_token if (hf_token and hf_token != "YOUR_HF_TOKEN") else None
    
    try:
        client = InferenceClient(model=model_endpoint, token=token_str)
        clean_name = predicted_disease.replace("___", " ").replace("_", " ")
        
        messages = [
            {
                "role": "system", 
                "content": "You are a senior plant pathologist. Output response structured inside clean markdown subsections without pleasantries."
            },
            {
                "role": "user", 
                "content": f"Provide an organic or chemical treatment report for crop fields diagnosed with leaf disease: {clean_name}."
            }
        ]
        
        response = client.chat_completion(messages=messages, max_tokens=500, temperature=0.2)
        return response.choices.message.content
        
    except Exception as e:
        return "### 📋 Advisory Fallback\nApply broad-spectrum copper fungicide suspension."


def run_project_pipeline(epochs=5, model_type="MobileNetV2"):
    """Orchestrates ingestion pipelines, running model training, and validating states."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    train_loader, val_loader, test_loader, num_classes, class_names = get_desktop_data_loaders()
    if train_loader is None:
        print(" Data Error: Verify directory paths live safely inside 'Desktop/finalproject'")
        return

    if model_type == "Baseline":
        model = BaselineCNN(num_classes=num_classes)
    else:
        model = load_transfer_model(model_name=model_type, num_classes=num_classes)
        
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    best_val_loss = float('inf')
    desktop_path = os.path.expanduser("~/Desktop")
    project_dir = os.path.join(desktop_path, "finalproject")
    
    print(f"\n🚀 --- INITIATING PIPELINE RUN FOR MODEL: {model_type} ---")
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            
        scheduler.step()
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                
        val_loss /= len(val_loader.dataset)
        print(f"Epoch [{epoch+1}/{epochs}] | Validation Evaluation Loss: {val_loss:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(project_dir, f'best_{model_type.lower()}_model.pth'))

    print("\n🔍 --- TRAINING COMPLETE: RUNNING POST-TRAINING DEPLOYMENT ON UNBIASED TEST SPLIT ---")
    model.load_state_dict(torch.load(os.path.join(project_dir, f'best_{model_type.lower()}_model.pth')))
    comprehensive_eval(model, test_loader, device, model_name=model_type, classes=class_names)
    
    sample_class = class_names[0] if isinstance(class_names, list) and len(class_names) > 0 else "Tomato___Early_blight"
    print(f"\n🤖 --- STEP 6: AI GENERATIVE ADVISORY DEMO FOR [{sample_class}] ---")
    print(query_treatment_advisory(sample_class))


if __name__ == "__main__":
    run_project_pipeline(epochs=5, model_type="MobileNetV2")



📊 --- KAGGLE NEW PLANT DISEASES DATASET EDA ---
📈 Training (70%) Split: Found 38 classes containing 70295 total leaf images.
   ↳ Class Size Range: Min=1642 | Max=2022 images per folder.
📈 Validation (15%) Split: Found 38 classes containing 17572 total leaf images.
📈 Testing (15%) Split: Found 33 classes containing 0 total leaf images.

⚠️  [PIPELINE NOTICE]: Empty/missing 'test' class taxonomy detected.
   ↳ Duplicating 'valid' partition arrays to bypass testing target generation crash.

🚀 --- INITIATING PIPELINE RUN FOR MODEL: MobileNetV2 ---


C:\Users\pc\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:1095: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [ ]:
creating the sample outputs and predicted queue + hugging face-generated reply.

In [ ]:
import os
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from huggingface_hub import InferenceClient

def query_treatment_advisory(predicted_disease, hf_token=None):
    if "healthy" in predicted_disease.lower():
        return "### Leaf Status: Healthy\nNo pathogenic targets found. Maintain standard irrigation parameters."
    model_endpoint = "Qwen/Qwen2.5-7B-Instruct"
    token_str = hf_token if (hf_token and hf_token != "YOUR_HF_TOKEN") else None
    try:
        client = InferenceClient(model=model_endpoint, token=token_str)
        clean_name = predicted_disease.replace("___", " ").replace("_", " ")
        messages = [
            {"role": "system", "content": "You are a senior plant pathologist. Output response structured inside clean markdown subsections without pleasantries."},
            {"role": "user", "content": f"Provide an organic or chemical treatment report for crop fields diagnosed with leaf disease: {clean_name}."}
        ]
        response = client.chat_completion(messages=messages, max_tokens=500, temperature=0.2)
        return response.choices[0].message.content
    except Exception as e:
        return "### 📋 Advisory Fallback\nApply broad-spectrum copper fungicide suspension."

if __name__ == "__main__":
    desktop_path = os.path.expanduser("~/Desktop")
    project_dir = os.path.join(desktop_path, "finalproject")
    
    image_path = os.path.join(project_dir, "leaf.jpg")
    model_path = os.path.join(project_dir, "best_mobilenetv2_model.pth")
    output_txt_path = os.path.join(project_dir, "sample_outputs.txt")
    
    
    class_names = [
        "Apple___Apple_scab", "Apple___Black_rot", "Apple___Cedar_apple_rust", "Apple___healthy",
        "Blueberry___healthy", "Cherry_(including_sour)___Powdery_mildew", "Cherry_(including_sour)___healthy",
        "Corn___Cercospora_leaf_spot_Gray_leaf_spot", "Corn___Common_rust", "Corn___Northern_Leaf_Blight", "Corn___healthy",
        "Grape___Black_rot", "Grape___Esca_(Black_Measles)", "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy",
        "Orange___Haunglongbing_(Citrus_greening)", "Peach___Bacterial_spot", "Peach___healthy",
        "Pepper,_bell___Bacterial_spot", "Pepper,_bell___healthy", "Potato___Early_blight", "Potato___Late_blight", "Potato___healthy",
        "Raspberry___healthy", "Soybean___healthy", "Squash___Powdery_mildew", "Strawberry___Leaf_scorch", "Strawberry___healthy",
        "Tomato___Bacterial_spot", "Tomato___Early_blight", "Tomato___Late_blight", "Tomato___Leaf_Mold",
        "Tomato___Septoria_leaf_spot", "Tomato___Spider_mites_Two-spotted_spider_mite", "Tomato___Target_Spot",
        "Tomato___Tomato_Yellow_Leaf_Curl_Virus", "Tomato___Tomato_mosaic_virus", "Tomato___healthy"
    ]
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("⏳ Starting live sample inference generation...")

    
    model = models.mobilenet_v2(weights=None)
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(model.last_channel, len(class_names))
    )
    

    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.to(device)
        model.eval()
        print("✔️ Successfully loaded trained weights from checkpoint file.")
    else:
        raise FileNotFoundError(f"Missing weight checkpoint target file at: {model_path}")

    val_test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    if os.path.exists(image_path):
        raw_img = Image.open(image_path).convert("RGB")
        input_tensor = val_test_transform(raw_img).unsqueeze(0).to(device)
    else:
        raise FileNotFoundError(f"Missing sample source target image at: {image_path}")

    with torch.no_grad():
        outputs = model(input_tensor)
        _, predicted_idx = torch.max(outputs, 1)
        predicted_class = class_names[predicted_idx.item()]
        
    print(f"🎯 Model Prediction Complete -> Detected Label: {predicted_class}")

    print("🌐 Contacting Hugging Face Qwen inference nodes for treatment guidance...")
    ai_response = query_treatment_advisory(predicted_class, hf_token="YOUR_HF_TOKEN")

    log_content = f"""===========================================================================
📋 SAMPLE INFERENCE PIPELINE LOG (PROJECT DELIVERABLE)
===========================================================================
Target Test Input Image File: leaf.jpg
Trained Weight Engine Baseline: best_mobilenetv2_model.pth

[STAGE 1: CNN COMPUTER VISION DIAGNOSIS]
↳ Predicted Class Index: {predicted_idx.item()}
↳ Predicted Queue Label: {predicted_class}

[STAGE 2: HUGGING FACE QWEN GENERATED TREATMENT REPLY]
{ai_response}
===========================================================================
"""

    with open(output_txt_path, "w", encoding="utf-8") as f:
        f.write(log_content)
        
    print(f"\n💾 Deliverable Generated! 'sample_outputs.txt' saved at:\n👉 {output_txt_path}\n")
    print(log_content)

⏳ Starting live sample inference generation...
✔️ Successfully loaded trained weights from checkpoint file.
🎯 Model Prediction Complete -> Detected Label: Pepper,_bell___healthy
🌐 Contacting Hugging Face Qwen inference nodes for treatment guidance...

💾 Deliverable Generated! 'sample_outputs.txt' saved at:
👉 C:\Users\pc/Desktop\finalproject\sample_outputs.txt

📋 SAMPLE INFERENCE PIPELINE LOG (PROJECT DELIVERABLE)
Target Test Input Image File: leaf.jpg
Trained Weight Engine Baseline: best_mobilenetv2_model.pth

[STAGE 1: CNN COMPUTER VISION DIAGNOSIS]
↳ Predicted Class Index: 19
↳ Predicted Queue Label: Pepper,_bell___healthy

[STAGE 2: HUGGING FACE QWEN GENERATED TREATMENT REPLY]
### Leaf Status: Healthy
No pathogenic targets found. Maintain standard irrigation parameters.

